# 4. Gaussian Process Regression

## Baseline GPR

Designing a MEMS backplate requires tuning its fundamental resonance frequency $f_0$ while balancing acoustic performance, mechanics, and fabrication limits. Because FEM simulations are computationally expensive, we use Gaussian Process Regression (GPR) as a surrogate model to directly learn the forward mapping from geometry to frequency:

$$(a, h, p, d) \;\xrightarrow{\quad\mathcal{GP}\quad}\; f_0$$

**Design Parameters & Fabrication Constraints**

The plate geometry is governed by four continuous parameters:
* $a$: Plate half-side length
* $h$: Plate thickness
* $p$: Perforation pitch (hole-to-hole center distance)
* $d$: Perforation hole diameter

A viable microphone plate must satisfy strict physical validity and fabrication rules:

* Thin-Plate Assumption ($h/p < 2$)
* Perforation Ratio ($R = (d/p)^2 < 0.5$)
* Lithographic Clearance ($p \ge d + 5\text{ um}$): Preserves a minimum sidewall ligament of $5\text{ um}$ between adjacent etch holes to prevent sidewall collapse during release etching.

![Geometry](geom.png)


You can play around with setting of variables `N_TRAIN` and `N_CANDIDATES`:

1. `N_TRAIN` (High-Fidelity Physics Evaluations)
* Meaning: The number of calls made to the ground-truth physical solver.

* Computational Cost: EXTREMELY EXPENSIVE. In real engineering workflows, each run  represents a 3D finite-element solve taking minutes to hours.

* Engineering Goal: Keep N_TRAIN as LOW as possible to minimize total solve time.

2. `N_CANDIDATES` (Surrogate Query Pool)
* Meaning: The number of virtual geometry designs evaluated ONLY by the trained Gaussian Process surrogate model (gp.predict).

* Computational Cost: VIRTUALLY FREE. Running vectorized matrix math over 50,000 points takes a fraction of a second in NumPy/scikit-learn. No physics solver is executed.

* Engineering Goal: Keep N_CANDIDATES HIGH (50,000 to 100,000). A dense candidate cloud provides fine geometric resolution across the 4D parameter space (a, h, p, d), allowing instant inversion and target lookup via np.argmin without running numerical optimizers.

**Goal:** Your task is to answer *How Low Can We Go?* and minimize the number of calls of computationally expensive simulation calls `get_freq()`.







In [ ]:
# import necessary libraries
import numpy as np
import matplotlib.pyplot as plt
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import Matern, ConstantKernel as C

from gp_utils import get_freq

### Configuration and Parameter Bounds


In [ ]:
TARGET_FREQ = 45000.0  # target fundamental frequency in Hz (adjustable)
target_khz = TARGET_FREQ / 1e3

# Parameter bounds:
BOUNDS = np.array([
    [0.5e-3, 2.0e-3],    # a: (half-side, m)
    [5.0e-6, 20.0e-6],    # h: (thickness, m)
    [20.0e-6, 50.0e-6],  # p: (pitch, m)
    [2.0e-6, 10.0e-6]     # d: (diameter, m)
])

lower_b = BOUNDS[:, 0]
upper_b = BOUNDS[:, 1]

def to_unit(x):
    """Normalize physical parameters into [0, 1]."""
    return (x - lower_b) / (upper_b - lower_b)

def from_unit(u):
    """Unscale unit coordinates back to physical units."""
    return lower_b + u * (upper_b - lower_b)

def sample_valid_designs(n_samples, rng):
    # Samples n_init points uniformly across the parameter box [a, h, p, d] 
    # using random scaling between the lower and upper bounds (rng)

    u = rng.rand(n_samples, 4)
    x = from_unit(u)
    # Ensure R = (d/p)^2 < 0.5
    x[:, 3] = np.minimum(x[:, 3], x[:, 2] * np.sqrt(0.49))
    return x


### Initial Dataset Generation and GPR Setup


In [ ]:
# TODO: play around with these numbers
N_TRAIN = 10
N_CANDIDATES = 50000

# Fix random seed for deterministic sampling and kernel optimization
rng = np.random.RandomState(42)

# ==============================================================================
# Initial Design Generation & Physics Evaluation
# ==============================================================================
# Generate initial N_TRAIN points within bounds
X_train_phys = sample_valid_designs(N_TRAIN, rng)

# Evaluate exact fundamental natural frequency f_0 (scaled to kHz for numerical stability)
y_train_khz = np.array([
    get_freq(row[0], row[1], row[2], row[3], m=1, n=1) / 1e3
    for row in X_train_phys
])

# ==============================================================================
# Gaussian Process Fit (Normalized Design Space [0, 1]^4)
# ==============================================================================

kernel = C(1.0, (1e-2, 1e4)) * Matern(length_scale=[0.5, 0.5, 0.5, 0.5], nu=2.5)
gp = GaussianProcessRegressor(
    kernel=kernel, 
    alpha=1e-4, 
    n_restarts_optimizer=10, 
    random_state=42
)
gp.fit(to_unit(X_train_phys), y_train_khz)

# ==============================================================================
# Candidate Space & Surrogate Inversion
# ==============================================================================
# Generate a dense candidate pool and project into normalized coordinates
candidates_phys = sample_valid_designs(N_CANDIDATES, rng)
candidates_u = to_unit(candidates_phys)

# Compute posterior mean (mu) and standard deviation (sigma) across all candidates
pred_f_khz, pred_std_khz = gp.predict(candidates_u, return_std=True)

# Select the candidate whose posterior mean is closest to the target frequency
error_khz = np.abs(pred_f_khz - target_khz)

# Select the candidate whose posterior mean is closest to the target frequency
best_idx = np.argmin(error_khz + 0.5 * pred_std_khz)
pred_target_khz = pred_f_khz[best_idx]
pred_target_std = pred_std_khz[best_idx]

# Ground-Truth Validation of the Initial Surrogate Recommendation
best_params = candidates_phys[best_idx]
true_freq = get_freq(*best_params, m=1, n=1)
rel_error = abs(true_freq - TARGET_FREQ) / TARGET_FREQ * 100

print("Optimal Design Found:")
print(f"  Plate Half-side a:    {best_params[0]*1e6:.3f} um")
print(f"  Thickness h:          {best_params[1]*1e6:.2f} um")
print(f"  Pitch p:              {best_params[2]*1e6:.2f} um")
print(f"  Diameter d:           {best_params[3]*1e6:.2f} um")
print(f"  GP Predicted Freq:    {pred_target_khz:.2f} +/- {2*pred_target_std:.2f} kHz")
print(f"  Actual Physics Freq:  {true_freq/1e3:.2f} kHz (Target: {target_khz:.2f} kHz)")
print(f"  Absolute Error:       {rel_error:.3f}%")

In [ ]:
param_labels = ['a / mm', 'h / um', 'p / um', 'd / um']
param_names = ['Plate Half-side a', 'Thickness h', 'Pitch p', 'Diameter d']
scale_factors = [1e3, 1e6, 1e6, 1e6]

plt.figure(figsize=(10, 5))

for i in range(4):
    plt.subplot(2, 2, i + 1)
    
    # 1D slice through the initial best surrogate design
    sweep_vals = np.linspace(BOUNDS[i, 0], BOUNDS[i, 1], 250)
    X_slice_phys = np.tile(best_params, (250, 1))
    X_slice_phys[:, i] = sweep_vals
    
    X_slice_u = to_unit(X_slice_phys)
    mu_khz, std_khz = gp.predict(X_slice_u, return_std=True)
    
    true_slice_khz = np.array([
        get_freq(row[0], row[1], row[2], row[3], m=1, n=1) / 1e3
        for row in X_slice_phys
    ])
    
    x_plot = sweep_vals * scale_factors[i]
    x_opt = best_params[i] * scale_factors[i]
    
    # Model predictions and true 1D physics slice
    plt.plot(x_plot, true_slice_khz, 'k--', linewidth=1.6, label='get_freq()')
    plt.plot(x_plot, mu_khz,  color='darkorange', linewidth=2.0, label='GP Posterior Mean')
    plt.fill_between(
        x_plot,
        mu_khz - 2 * std_khz,
        mu_khz + 2 * std_khz,
        color='orange',
        alpha=0.25,
        label=r"$\pm 2\sigma$"
    )
    
   
    plt.axhline(target_khz, color='crimson', linestyle=':', linewidth=1.5, label=f'Target ({target_khz:.1f} kHz)')
    plt.axvline(x_opt, color='darkgreen', linestyle='-', linewidth=2.0, label='Initial Best Guess')
    
    plt.title(f'Posterior $f \\mid \\mathcal{{D}}$ vs. {param_names[i]}')
    plt.xlabel(param_labels[i])
    plt.ylabel(r'$f_0$ / kHz')
    plt.grid(True, linestyle=':', alpha=0.6)
    
    if i == 1:
        plt.legend(loc='upper left', fontsize=8.5, framealpha=0.85)

plt.tight_layout()
plt.show()